# 04 - Gold Layer: Claims & Driving Behavior Aggregates

**Project:** Auto Insurance Claims & Telematics Analytics

## What this notebook does
Builds business-consumable gold tables from both independent silver tracks -
claims/fraud analysis from silver_claims, and driving behavior analysis from
silver_telematics_events. These remain two separate sets of gold tables,
consistent with the no-join decision made at the silver layer.

## Tables created (claims track)
- `main.auto_insurance_telematics.gold_claims_by_severity`
- `main.auto_insurance_telematics.gold_fraud_risk_summary`
- `main.auto_insurance_telematics.gold_fraud_risk_by_state`

## Tables created (telematics track)
- `main.auto_insurance_telematics.gold_driving_behavior_by_vehicle`
- `main.auto_insurance_telematics.gold_driving_event_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_claims_by_severity
COMMENT 'Claim volume and cost by incident severity. NOTE: pct_fraud_reported shows a strong correlation with severity (Major Damage: 60.5% vs Trivial Damage: 6.7%), consistent across both collision types within Major Damage (not a single-incident-type artifact). This dataset was originally built as a fraud-detection practice/training dataset, so this signal may be intentionally engineered into the synthetic data rather than reflecting real-world fraud patterns - worth noting when presenting this finding.'
AS
SELECT
  incident_severity,
  COUNT(*) AS claim_count,
  ROUND(AVG(total_claim_amount), 2) AS avg_claim_amount,
  SUM(total_claim_amount) AS total_claim_amount,
  ROUND(AVG(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100, 1) AS pct_fraud_reported
FROM main.auto_insurance_telematics.silver_claims
GROUP BY incident_severity
ORDER BY avg_claim_amount DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_fraud_risk_summary
COMMENT 'Fraud rate and claim behavior by policy state and insured occupation, for SIU/fraud review drill-down. CAUTION: 11 of 42 state x occupation groups (26%) have fewer than 20 claims - percentages in those rows are unreliable and can shift dramatically with just 1-2 claims. For a reliable headline fraud rate, see gold_fraud_risk_by_state instead.'
AS
SELECT
  policy_state,
  insured_occupation,
  COUNT(*) AS claim_count,
  ROUND(AVG(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100, 1) AS pct_fraud_reported,
  ROUND(AVG(total_claim_amount), 2) AS avg_claim_amount
FROM main.auto_insurance_telematics.silver_claims
GROUP BY policy_state, insured_occupation
ORDER BY pct_fraud_reported DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_fraud_risk_by_state
COMMENT 'Fraud rate by policy state only - reliable sample sizes (300+ claims per state), suitable for headline reporting. See gold_fraud_risk_summary for occupation-level drill-down, where roughly a quarter of state x occupation combinations have small sample sizes (under 20 claims) and should be read cautiously.'
AS
SELECT
  policy_state,
  COUNT(*) AS claim_count,
  ROUND(AVG(CASE WHEN fraud_reported = 'Y' THEN 1 ELSE 0 END) * 100, 1) AS pct_fraud_reported,
  ROUND(AVG(total_claim_amount), 2) AS avg_claim_amount
FROM main.auto_insurance_telematics.silver_claims
GROUP BY policy_state
ORDER BY pct_fraud_reported DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_driving_behavior_by_vehicle
COMMENT 'Event counts and average speed by vehicle and event type, from live telematics stream'
AS
SELECT
  vehicle_id,
  event_type,
  COUNT(*) AS event_count,
  ROUND(AVG(speed_mph), 1) AS avg_speed_mph
FROM main.auto_insurance_telematics.silver_telematics_events
GROUP BY vehicle_id, event_type
ORDER BY vehicle_id, event_count DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.auto_insurance_telematics.gold_driving_event_summary
COMMENT 'Fleet-wide event type distribution and average speed, from live telematics stream - reliable aggregate across all 25 simulated vehicles'
AS
SELECT
  event_type,
  COUNT(*) AS event_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total,
  ROUND(AVG(speed_mph), 1) AS avg_speed_mph
FROM main.auto_insurance_telematics.silver_telematics_events
GROUP BY event_type
ORDER BY event_count DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM main.auto_insurance_telematics.gold_driving_event_summary;

event_type,event_count,pct_of_total,avg_speed_mph
normal_driving,112,71.8,37.6
rapid_acceleration,14,9.0,42.2
hard_brake,13,8.3,37.7
speeding,11,7.1,76.8
harsh_cornering,6,3.8,35.2
